# IB Integration

In [7]:
ls

 Volume in drive C is Windows
 Volume Serial Number is 78FF-20E0

 Directory of c:\Uni\Year 3\flash-crash-sentinel\python_components\integrations\interactiveBrokers

27.12.2025  19:12    <DIR>          .
27.12.2025  19:02    <DIR>          ..
27.12.2025  19:15               420 IBConnect.py
28.12.2025  20:29             6�441 research.ipynb
               2 File(s)          6�861 bytes
               2 Dir(s)  139�968�921�600 bytes free


In [9]:
import tkinter as tk
from tkinter import ttk, messagebox
import threading
import time
from datetime import datetime
from collections import deque
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.patches import Rectangle
from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
import warnings
from services.vol_service import VolService

ModuleNotFoundError: No module named 'services'

In [ ]:
class IBApp(EWrapper, EClient):
    def __init__(self, callback = None):
        EClient.__init__(self, self)
        self.callback = callback
        self.connected = False
        self.last_price = None
        self.bid_price = None
        self.ask_price = None
        self.historical_data = {}
        self.histDone = threading.Event()

    def error(self, reqId, errorCode, errorString, advancedOrderRejectJson= ""):
        if errorCode in [2104, 2106, 2158, 2176]:
            return
        if errorCode == 10167:
            print("Note: Delayed market data.")
            return
        print(f"Error | reqID: {reqId} | errorCode: {errorCode}  \n Msg: {errorString}")

    def nextValidId(self, orderId):
        self.connected = True
        print(f"Connected.")
    

    def historicalData(self, reqId, bar):
        if reqId not in self.historical_data:
            self.historical_data[reqId] = []
        self.historical_data[reqId].append({
            "o": bar.open,
            "h": bar.high,
            "l": bar.low,
            "c": bar.close,            
        })

    def historicalDataEnd(self, reqId, start, end):
        self.histDone.set()

    def tickPrice(self, reqId, tickType, price, attrib):
        if price <= 0:
            return
        if tickType == 4:
            self.last_price = price
            if self.callback:
                self.callback("price",price, datetime.now())
        elif tickType == 1:
            self.bid_price = price
        elif tickType == 2:
            self.ask_price = price
        

NameError: name 'EWrapper' is not defined

In [ ]:
class OHLCBar():
    def __init__(self, timestamp, open):
        self.timestamp = timestamp
        self.open = open 
        self.high = open
        self.low = open
        self.close = open
        self.tick_count = 1
        self.regime = 0 # Vol. regime (0: low, 1: med, 2: high)


    def update(self, price):
        self.high = max(self.high, price)
        self.low = min(self.low, price)
        self.close = price
        self.tick_count += 1 

    @property 
    def volatility(self): # Replace with Yang-Zhang from vol_service.py
        return (self.high - self.low) / self.close if self.close > 0 else 0
